# Search engine




In [1]:
import os, sys, logging
logging.basicConfig(stream=sys.stdout, 
                    format='%(asctime)s %(message)s',
                    level=logging.INFO)

## load DBpedia data

In [2]:
from rdflib import URIRef

database_url = 'http://localhost:3030/dbpedia_en'
identifier = URIRef("https://mangosaurus.eu/dbpedia")

In [3]:
from rdflib.plugins.stores.sparqlstore import SPARQLUpdateStore
from nifigator import NifGraph

# Connect to triplestore
store = SPARQLUpdateStore(
    query_endpoint = database_url+'/sparql',
    update_endpoint = database_url+'/update'
)
# Create NifVectorGraph with this store
g = NifGraph(
    store=store, 
    identifier=identifier
)

Import the vector representations of the phrases, lemmas and contexts


In [4]:
import pickle

with open('..//data//dbpedia_phrases_10000.pickle', 'rb') as handle:
    v_phrases = pickle.load(handle)
with open('..//data//dbpedia_contexts_10000.pickle', 'rb') as handle:
    v_contexts = pickle.load(handle)
with open('..//data//dbpedia_lemmas_10000.pickle', 'rb') as handle:
    lemmas_dict = pickle.load(handle)

In [5]:
lang = "en"

stop_words_en = [
    'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of',
    'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during',
    'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off',
    'over', 'under', 'further',
    'per',
]

stop_words_nl = [
    'een', 'de', 'het', 'en', 'maar', 'als', 'of', 'omdat', 'van',
    'te', 'in', 'op', 'aan', 'met', 'voor', 'er', 'om', 'dan', 'of',
    'door', 'over', 'bij', 'ook', 'tot', 'uit', 'naar', 'want', 'nog',
    'toch', 'al', 'dus', 'onder', 'tegen', 'na', 'reeds',
]
# set the parameters to create the NifVector graph
params = {
    "words_filter": {
        "data": stop_words_en if lang=="en" else stop_words_nl,
        "name": "nifvec.stopwords"
    },
}

In [6]:
from selmr import SELMR, LanguageMultisets

# construct a SELMR data structure with the DBpedia phrases and contexts
selmr = SELMR(
    multisets=LanguageMultisets(v_phrases, v_contexts, lemmas_dict),
    params=params
)

In [7]:
selmr.most_similar("revealed", topcontexts=15)

Counter({'found': 15,
         'reported': 15,
         'revealed': 15,
         'determined': 14,
         'said': 14,
         'discovered': 13,
         'shown': 13,
         'stated': 13,
         'suggested': 13,
         'claimed': 13,
         'proposed': 13,
         'argued': 12,
         'held': 12,
         'noted': 11,
         'estimated': 11})

In [8]:
# # the vector of 'is' is a subset of the vector of 'be'
# print(phrases['is'] <= v_lemmas['be'])

# # the vector 'discovered' is a subset of the vector of 'discover'
# print(v_phrases['discovered'] <= v_lemmas['discover'])

# # the union of the vectors of 'king' and 'kings' is equal to the vector of 'king'
# print(v_phrases['kings'] + v_phrases['king'] == v_lemmas['king'])

### Load and set up two DBpedia pages

We take two pages from DBpedia about two stars, Aldebaran and Antares. 

In [9]:
# Read two documents in DBpedia about Aldebaran and Antares stars
doc1 = g.get(
    URIRef("http://dbpedia.org/resource/Aldebaran?dbpv=2020-07&nif=context")
)
doc2 = g.get(
    URIRef("http://dbpedia.org/resource/Antares?dbpv=2020-07&nif=context")
)

The first document reads:

In [10]:
print(doc1)

(nif:Context) uri = <http://dbpedia.org/resource/Aldebaran?dbpv=2020-07&nif=context>
  sourceUrl : <http://en.wikipedia.org/wiki/Aldebaran?oldid=964792900&ns=0>
  predLang : <http://lexvo.org/id/iso639-3/eng>
  isString : 'Aldebaran , designated α Tauri (Latinized to Alpha Tauri, abbreviated Alpha Tau, α Tau), is an orange giant star measured to be about 65 light-years from the Sun in the zodiac constellation Taurus. It is the brightest star in Taurus and generally the fourteenth-brightest star in the night sky, though it varies slowly in brightness between magnitude 0.75 and 0.95. Aldebaran is believed to host a planet several times the mass of Jupiter, named Aldebaran b. Aldebaran is a red giant, cooler than the sun with a surface temperature of 3,900 K, but its radius is about 44 times the sun\'s, so it is over 400 times as luminous. It spins slowly and takes 520 days to complete a rotation. The planetary exploration probe Pioneer 10 is heading in the general direction of the star a

```console
(nif:Context) uri = <http://dbpedia.org/resource/Aldebaran?dbpv=2020-07&nif=context>
  sourceUrl : <http://en.wikipedia.org/wiki/Aldebaran?oldid=964792900&ns=0>
  predLang : <http://lexvo.org/id/iso639-3/eng>
  isString : 'Aldebaran , designated α Tauri (Latinized to Alpha Tauri, abbreviated Alpha Tau, α Tau), is an orange giant star measured to be about 65 light-years from the Sun in the zodiac constellation Taurus. It is the brightest star in Taurus and generally the fourteenth-brightest star in the night sky, though it varies slowly in brightness between magnitude 0.75 and 0.95. Aldebaran is believed to host a planet several times the mass of Jupiter, named Aldebaran b. Aldebaran is a red giant, cooler than the sun with a surface temperature of 3,900 K, but its radius is about 44 times the sun\'s, so it is over 400 times as luminous. It spins slowly and takes 520 days to complete a rotation. The planetary exploration probe Pioneer 10 is heading in the general direction of the star and should make its closest approach in about two million years.\n\nNomenclature\nThe traditional name Aldebaran derives from the Arabic al Dabarān, meaning "the follower", because it seems to follow the Pleiades. In 2016, the I... '
  firstSentence : 'Aldebaran , designated α Tauri (Latinized to Alpha Tauri, abbreviated Alpha Tau, α Tau), is an orange giant star measured to be about 65 light-years from the Sun in the zodiac constellation Taurus.'
  lastSentence : '* Daytime occultation of Aldebaran by the Moon (Moscow, Russia) YouTube video'
```

THe second document reads:

In [11]:
print(doc2)

(nif:Context) uri = <http://dbpedia.org/resource/Antares?dbpv=2020-07&nif=context>
  sourceUrl : <http://en.wikipedia.org/wiki/Antares?oldid=964919229&ns=0>
  predLang : <http://lexvo.org/id/iso639-3/eng>
  isString : 'Antares , designated α Scorpii (Latinised to Alpha Scorpii, abbreviated Alpha Sco, α Sco), is on average the fifteenth-brightest star in the night sky, and the brightest object in the constellation of Scorpius. Distinctly reddish when viewed with the naked eye, Antares is a slow irregular variable star that ranges in brightness from apparent magnitude +0.6 to +1.6. Often referred to as "the heart of the scorpion", Antares is flanked by σ Scorpii and τ Scorpii near the center of the constellation. Classified as spectral type M1.5Iab-Ib, Antares is a red supergiant, a large evolved massive star and one of the largest stars visible to the naked eye. Its exact size remains uncertain, but if placed at the center of the Solar System, it would reach to somewhere between the orb

```console
(nif:Context) uri = <http://dbpedia.org/resource/Antares?dbpv=2020-07&nif=context>
  sourceUrl : <http://en.wikipedia.org/wiki/Antares?oldid=964919229&ns=0>
  predLang : <http://lexvo.org/id/iso639-3/eng>
  isString : 'Antares , designated α Scorpii (Latinised to Alpha Scorpii, abbreviated Alpha Sco, α Sco), is on average the fifteenth-brightest star in the night sky, and the brightest object in the constellation of Scorpius. Distinctly reddish when viewed with the naked eye, Antares is a slow irregular variable star that ranges in brightness from apparent magnitude +0.6 to +1.6. Often referred to as "the heart of the scorpion", Antares is flanked by σ Scorpii and τ Scorpii near the center of the constellation. Classified as spectral type M1.5Iab-Ib, Antares is a red supergiant, a large evolved massive star and one of the largest stars visible to the naked eye. Its exact size remains uncertain, but if placed at the center of the Solar System, it would reach to somewhere between the orbits of Mars and Jupiter. Its mass is calculated to be around 12 times that of the Sun. Antares is the brightest, most massive, and most evolved stellar member of the nearest OB association, the Scorpius–Centaurus Associ... '
  firstSentence : 'Antares , designated α Scorpii (Latinised to Alpha Scorpii, abbreviated Alpha Sco, α Sco), is on average the fifteenth-brightest star in the night sky, and the brightest object in the constellation of Scorpius.'
  lastSentence : '* Best Ever Image of a Star’s Surface and Atmosphere - First map of motion of material on a star other than the Sun'
```

## Find similar sentences

For sentences similarities we sum the contexts of the all the phrases in the sentences, thereby obtaining a multiset representation of the sentence. Then we calculate the Jaccard distance between the sentences and sort with increasing distance.

The Jaccard index is

```{math}
J(A, B) = \frac { | A \bigcap B |} { |A \bigcup B| }
```

Create a vector of every sentences of both documents.


In [12]:
# setup dictionary with sentences and their vector representation
doc1_vector = {
    sent.anchorOf: selmr.derive_multisets(
        documents=[sent.anchorOf],
        merge_dict=True
    )
    for sent in doc1.sentences
}
doc2_vector = {
    sent.anchorOf: selmr.derive_multisets(
        documents=[sent.anchorOf],
        merge_dict=True
    )
    for sent in doc2.sentences
}

Calculate the distances (based on Jaccard index) of all sentence combinations of first and second document.


In [13]:
from nifigator.multisets import jaccard_index, merge_multiset

# Calculate the Jaccard distance for all sentence combinations
d = {
    (s1, s2): 1 - jaccard_index(c1, c2)
    for s1, c1 in doc1_vector.items()
    for s2, c2 in doc2_vector.items()
}
# Sort the results with lowest distance
similarities = sorted(d.items(), key=lambda item: item[1])

Print the results


In [27]:
# print the results
for item in similarities[0:5]:
    print(repr(item[0][0]) + " \n<- distance = "+str(item[1])+' = {0:.4f}'.format(float(item[1]))+" ->\n"+repr(item[0][1])+"\n")

'References' 
<- distance = 0 = 0.0000 ->
'References'

'External links' 
<- distance = 0 = 0.0000 ->
'External links'

'It is a variable star listed in the General Catalogue of Variable Stars, but it is listed using its Bayer designation and does not have a separate variable star designation.' 
<- distance = 112/425 = 0.2635 ->
'Antares is a variable star and is listed in the General Catalogue of Variable Stars but as a Bayer-designated star it does not have a separate variable star designation.'

'In 2016, the International Astronomical Union Working Group on Star Names (WGSN) approved the proper name Aldebaran for this star.' 
<- distance = 131/379 = 0.3456 ->
'In 2016, the International Astronomical Union organised a Working Group on Star Names (WGSN) to catalog and standardise proper names for stars.'

'Aldebaran is 5.47 degrees south of the ecliptic and so can be occulted by the Moon.' 
<- distance = 203/352 = 0.5767 ->
'Occultations and conjunctions\nAntares is 4.57 degrees sout

```console
'References' 
<- distance = 0 = 0.0000 ->
'References'

'External links' 
<- distance = 0 = 0.0000 ->
'External links'

'It is a variable star listed in the General Catalogue of Variable Stars, but it is listed using its Bayer designation and does not have a separate variable star designation.' 
<- distance = 112/425 = 0.2635 ->
'Antares is a variable star and is listed in the General Catalogue of Variable Stars but as a Bayer-designated star it does not have a separate variable star designation.'

'In 2016, the International Astronomical Union Working Group on Star Names (WGSN) approved the proper name Aldebaran for this star.' 
<- distance = 131/379 = 0.3456 ->
'In 2016, the International Astronomical Union organised a Working Group on Star Names (WGSN) to catalog and standardise proper names for stars.'

'Aldebaran is 5.47 degrees south of the ecliptic and so can be occulted by the Moon.' 
<- distance = 203/352 = 0.5767 ->
'Occultations and conjunctions\nAntares is 4.57 degrees south of the ecliptic, one of four first magnitude stars within 6° of the ecliptic (the others are Spica, Regulus and Aldebaran), so it can be occulted by the Moon.'
```

## Explainable text search

For text search we need another distance function. Now we are interested in the extent to which the vector of a sentence contains the vector representation of a query. For this we use the containment or support, defined by the cardinality of the intersection between A and B divided by the cardinality of A.

```{math}
containment(A, B) = \frac { | A \bigcap B |} { |A| }
```

The sentence with the highest containment has the most contexts in common and thus is the closest to the text.

In [15]:
# setup dictionary with sentences and their contexts
v_doc_phrases = {
    sent.anchorOf: selmr.derive_multisets(
        documents=[sent.anchorOf], 
    )
    for sent in doc1.sentences+doc2.sentences
}
v_doc_lemmas = {
    sent.anchorOf: selmr.derive_multisets(
        documents=[sent.anchorOf], 
        lemmas=True
    )
    for sent in doc1.sentences+doc2.sentences
}

### Using the MinHashSearch 


In [16]:
# from selmr import MinHashSearch

# mhs = MinHashSearch(
#     selmr=selmr,
#     documents=v_doc_lemmas
# )

E:\code\python\nlp\lib\site-packages\datasketch\lshensemble_partition.py:72: RuntimeWarning: invalid value encountered in divide
  return np.sum((float(sizes[u])-sizes[l:u+1])/float(sizes[u])*counts[l:u+1])


In [17]:
# import pickle 

# with open('..\\data\\minhash.pickle', 'wb') as fh:
#     pickle.dump(mhs.minhash_dict, fh)

In [18]:
with open('..\\data\\minhash.pickle', 'rb') as fh:
    minhash_dict = pickle.load(fh)

In [19]:
from selmr import MinHashSearch

mhs = MinHashSearch(
    selmr=selmr,
    minhash_dict=minhash_dict,
    documents=v_doc_lemmas,
)

In [20]:
from nifigator import jaccard_index

s1 = 'large'
s2 = 'small'
print("estimated Jaccard index: "+str(mhs.minhash_dict[s2].jaccard(mhs.minhash_dict[s1])))
print("actual Jaccard index: "+str(float(jaccard_index(
    set(p[0] for p in v_phrases[s1].most_common(15)),
    set(p[0] for p in v_phrases[s2].most_common(15)),
))))

estimated Jaccard index: 0.390625
actual Jaccard index: 0.36363636363636365


```console
estimated Jaccard index: 0.390625
actual Jaccard index: 0.36363636363636365
```

In [21]:
# s1 = 'was'
# s2 = 'be'
# print("estimated Jaccard index: "+str(mhs.minhash_dict[s2].jaccard(mhs.minhash_dict[s1])))
# print("actual Jaccard index: "+str(float(jaccard_index(
#     set(p[0] for p in v_lemmas[s1].most_common(15)),
#     set(p[0] for p in v_lemmas[s2].most_common(15)),
# ))))

```console
estimated Jaccard index: 1.0
actual Jaccard index: 1.0
```

In [33]:
s1 = 'Antares is a variable star and is listed in the General Catalogue of Variable Stars but as a Bayer-designated star it does not have a separate variable star designation.'
s2 = 'It is a variable star listed in the General Catalogue of Variable Stars, but it is listed using its Bayer designation and does not have a separate variable star designation.'

print("estimated Jaccard index: "+str(mhs.minhash_documents[s2].jaccard(mhs.minhash_documents[s1])))
print("actual Jaccard index: "+str(float(jaccard_index(
    merge_multiset(v_doc_phrases[s1]).keys(),
    merge_multiset(v_doc_phrases[s2]).keys()
))))

estimated Jaccard index: 0.6953125
actual Jaccard index: 0.7364705882352941


```console
estimated Jaccard index: 0.6953125
actual Jaccard index: 0.7364705882352941
```

In [23]:
query = "the brightest star in the constellation of Taurus"
scores = mhs.get_scores([query], selmr=selmr)
for item, distance in list(scores.items())[0:3]:
    print(repr(item) +': {0:.4f}'.format(float(distance)))

'Aldebaran is the brightest star in the constellation Taurus and so has the Bayer designation α Tauri, Latinised as Alpha Tauri.': 0.0000
"It is the brightest star in Taurus and generally the fourteenth-brightest star in the night sky, though it varies slowly in brightness between magnitude 0.75 and 0.95. Aldebaran is believed to host a planet several times the mass of Jupiter, named Aldebaran b. Aldebaran is a red giant, cooler than the sun with a surface temperature of 3,900 K, but its radius is about 44 times the sun's, so it is over 400 times as luminous.": 0.1233
'As the brightest star in a Zodiac constellation, it is also given great significance within astrology.': 0.1644


```console
'Aldebaran is the brightest star in the constellation Taurus and so has the Bayer designation α Tauri, Latinised as Alpha Tauri.': 0.0000
"It is the brightest star in Taurus and generally the fourteenth-brightest star in the night sky, though it varies slowly in brightness between magnitude 0.75 and 0.95. Aldebaran is believed to host a planet several times the mass of Jupiter, named Aldebaran b. Aldebaran is a red giant, cooler than the sun with a surface temperature of 3,900 K, but its radius is about 44 times the sun's, so it is over 400 times as luminous.": 0.1233
'As the brightest star in a Zodiac constellation, it is also given great significance within astrology.': 0.1644
```

In [24]:
query = "the sun in the Taurus cluster"
scores = mhs.get_scores([query], selmr=selmr)
for item, distance in list(scores.items())[0:3]:
    print(repr(item) +': {0:.4f}'.format(float(distance)))

"It is the brightest star in Taurus and generally the fourteenth-brightest star in the night sky, though it varies slowly in brightness between magnitude 0.75 and 0.95. Aldebaran is believed to host a planet several times the mass of Jupiter, named Aldebaran b. Aldebaran is a red giant, cooler than the sun with a surface temperature of 3,900 K, but its radius is about 44 times the sun's, so it is over 400 times as luminous.": 0.1905
'Aldebaran , designated α Tauri (Latinized to Alpha Tauri, abbreviated Alpha Tau, α Tau), is an orange giant star measured to be about 65 light-years from the Sun in the zodiac constellation Taurus.': 0.2857
'Aldebaran is the brightest star in the constellation Taurus and so has the Bayer designation α Tauri, Latinised as Alpha Tauri.': 0.4762


```console
"It is the brightest star in Taurus and generally the fourteenth-brightest star in the night sky, though it varies slowly in brightness between magnitude 0.75 and 0.95. Aldebaran is believed to host a planet several times the mass of Jupiter, named Aldebaran b. Aldebaran is a red giant, cooler than the sun with a surface temperature of 3,900 K, but its radius is about 44 times the sun's, so it is over 400 times as luminous.": 0.1905
'Aldebaran , designated α Tauri (Latinized to Alpha Tauri, abbreviated Alpha Tau, α Tau), is an orange giant star measured to be about 65 light-years from the Sun in the zodiac constellation Taurus.': 0.2857
'Aldebaran is the brightest star in the constellation Taurus and so has the Bayer designation α Tauri, Latinised as Alpha Tauri.': 0.4762
```

In [25]:
query = "astronomer William Herschel discovered to Aldebaran"
scores = mhs.get_scores([query], selmr=selmr)
for item, distance in list(scores.items())[0:3]:
    print(repr(item) +': {0:.4f}'.format(float(distance)))

'English astronomer William Herschel discovered a faint companion to Aldebaran in 1782; an 11th magnitude star at an angular separation of 117″.': 0.0000
'It was then observed by Scottish astronomer James William Grant FRSE while in India on 23 July 1844.': 0.5412
"Follow on measurements of proper motion showed that Herschel's companion was diverging from Aldebaran, and hence they were not physically connected.": 0.6118


```console
'English astronomer William Herschel discovered a faint companion to Aldebaran in 1782; an 11th magnitude star at an angular separation of 117″.': 0.0000
'It was then observed by Scottish astronomer James William Grant FRSE while in India on 23 July 1844.': 0.5412
"Follow on measurements of proper motion showed that Herschel's companion was diverging from Aldebaran, and hence they were not physically connected.": 0.6118
```

In [37]:
i = mhs.matches(
   ["astronomer William Herschel revealed Aldebaran"],
   ['English astronomer William Herschel discovered a faint companion to Aldebaran in 1782; an 11th magnitude star at an angular separation of 117″.']
)
print("Score: "+str(i.score))
print("Full matches:")
for key, values in i.full_matches.items():
    for value in values:
        print((key, value[0]))
print("Close matches:")
for key, values in i.close_matches.items():
    for value in values:
        print((key, value[0], "{0:.4f}".format(float(value[1]))))


Score: 11/85
Full matches:
('astronomer', 'astronomer')
('astronomer William', 'astronomer William')
('astronomer William Herschel', 'astronomer William Herschel')
('William', 'William')
('William Herschel', 'William Herschel')
('Herschel', 'Herschel')
('Aldebaran', 'Aldebaran')
Close matches:
('revealed', 'discovered', '0.7333')


```console
Score: 11/85
Full matches:
('astronomer', 'astronomer')
('astronomer William', 'astronomer William')
('astronomer William Herschel', 'astronomer William Herschel')
('William', 'William')
('William Herschel', 'William Herschel')
('Herschel', 'Herschel')
('Aldebaran', 'Aldebaran')
Close matches:
('revealed', 'discovered', '0.7333')
```